Authorization and initializatin function

In [18]:
from Google import Create_Service

# Configuration Definitions
CLIENT_SECRET_FILE = 'client_secret.json'
API_NAME = 'sheets'
API_VERSION = 'v4'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

# Global Target parameters
SPREADSHEET_ID = "1btqCNcTbWSyF8ahVr2DTOcLVukne5nWTUWTLN9xePGU" # <-- Swap with your working spreadsheet ID

# Connect to Google
service = Create_Service(CLIENT_SECRET_FILE, API_NAME, API_VERSION, scopes=SCOPES)
sheets = service.spreadsheets()

🎉 sheets vv4 service created successfully.


Testing commands response

Fetch

In [10]:
response = sheets.values().get(
    spreadsheetId=SPREADSHEET_ID,
    range="Sheet1!A1:F5"
).execute()

rows = response.get('values', [])
print("Data Fetched:")
for row in rows:
    print(row)



Data Fetched:
['Lead Name', 'Email', 'Phone', 'Company', 'Lead Source', 'Status']


Update

In [11]:
# Let's set up column headers explicitly
headers_payload = {
    "values": [["Lead Name", "Email", "Phone", "Company", "Lead Source", "Status"]]
}

sheets.values().update(
    spreadsheetId=SPREADSHEET_ID,
    range="Sheet1!A1:F1",
    valueInputOption="USER_ENTERED",
    body=headers_payload
).execute()
print("✅ Headers successfully forced into Row 1.")

✅ Headers successfully forced into Row 1.


Batch Insert (basically batchUpdate for different ranges)

In [12]:
# 2. Structure your batch data payload
# We are pushing two distinct data arrays to separate parts of the workbook simultaneously
batch_insert_payload = {
    "valueInputOption": "USER_ENTERED",  # Parses formats natively like dates, numbers, currencies
    "data": [
        {
            "range": "Sheet1!A2:F3",      # Target Range A
            "values": [
                ["Natasha Romanoff", "natasha@shield.gov", "555-0300", "S.H.I.E.L.D.", "cold_outreach", "new"],
                ["Clint Barton", "hawkeye@avengers.org", "555-0400", "Avengers Academy", "referral", "contacted"]
            ]
        },
        {
            "range": "Sheet1!A4:F4",      # Target Range B
            "values": [
                ["Wanda Maximoff", "wanda@westview.io", "555-0500", "Independent", "website", "qualified"]
            ]
        }
    ]
}

# 3. Execute the batch operation
try:
    response = sheets.values().batchUpdate(
        spreadsheetId=SPREADSHEET_ID,
        body=batch_insert_payload
    ).execute()
    
    print("🎉 Batch data insertion completed successfully!")
    print(f"Total cells updated: {response.get('totalUpdatedCells')}")
    
except Exception as e:
    print(f"❌ Batch insertion failed: {e}")

🎉 Batch data insertion completed successfully!
Total cells updated: 18


Appending (Adding New Rows)

In [13]:
new_leads = {
    "values": [
        ["Tony Stark", "tony@stark.com", "555-0100", "Stark Industries", "linkedin", "new"],
        ["Bruce Banner", "hulk@avengers.org", "555-0200", "Gamma Labs", "website", "qualified"]
    ]
}

sheets.values().append(
    spreadsheetId=SPREADSHEET_ID,
    range="Sheet1!A:F",
    valueInputOption="USER_ENTERED",
    insertDataOption="INSERT_ROWS",
    body=new_leads
).execute()
print("✅ 2 New leads appended directly below existing records.")

✅ 2 New leads appended directly below existing records.


Batch Fetch (Read Multiple Separate Ranges at Once)

In [14]:
batch_fetch_response = sheets.values().batchGet(
    spreadsheetId=SPREADSHEET_ID,
    ranges=["Sheet1!A1:B1", "Sheet1!E2:F3"]
).execute()

value_ranges = batch_fetch_response.get('valueRanges', [])
for vr in value_ranges:
    print(f"Range: {vr.get('range')} -> Values: {vr.get('values')}")

Range: Sheet1!A1:B1 -> Values: [['Lead Name', 'Email']]
Range: Sheet1!E2:F3 -> Values: [['cold_outreach', 'new'], ['referral', 'contacted']]


Batch Update (Write to Multiple Ranges at Once)

In [15]:
batch_update_payload = {
    "valueInputOption": "USER_ENTERED",
    "data": [
        {"range": "Sheet1!F2", "values": [["contacted"]]}, # Update Tony's pipeline status
        {"range": "Sheet1!F3", "values": [["nurturing"]]}  # Update Bruce's pipeline status
    ]
}

sheets.values().batchUpdate(
    spreadsheetId=SPREADSHEET_ID,
    body=batch_update_payload
).execute()
print("✅ Batch update operations processed across distinct ranges successfully.")

✅ Batch update operations processed across distinct ranges successfully.


Data Validation

In [19]:
TARGET_TAB_ID = 0
def apply_dropdown_validation(sheet_id, start_row, end_row, start_col, end_col, allowed_list):
    """Creates a strict UI interactive dropdown menu for specific coordinate cells."""
    
    # Map raw text values into the explicit JSON syntax Google expects
    condition_values = [{"userEnteredValue": item} for item in allowed_list]
    
    request_body = {
        "requests": [
            {
                "setDataValidation": {
                    "range": {
                        "sheetId": sheet_id,
                        "startRowIndex": start_row,
                        "endRowIndex": end_row,
                        "startColumnIndex": start_col,
                        "endColumnIndex": end_col
                    },
                    "rule": {
                        "condition": {
                            "type": "ONE_OF_LIST",
                            "values": condition_values
                        },
                        "inputMessage": f"Choose an authorized option: {', '.join(allowed_list)}",
                        "strict": True,       # True rejects bad inputs completely
                        "showCustomUi": True   # Displays the dropdown arrow in the cell
                    }
                }
            }
        ]
    }
    
    sheets.batchUpdate(spreadsheetId=SPREADSHEET_ID, body=request_body).execute()
    print("✅ Dropdown validation applied successfully.")

# Example Test: Apply strict Status dropdown to Column F (Index 5), rows 2 through 20
apply_dropdown_validation(
    sheet_id=TARGET_TAB_ID,
    start_row=1, end_row=20, 
    start_col=5, end_col=6, 
    allowed_list=["new", "contacted", "qualified", "nurturing", "lost"]
)

✅ Dropdown validation applied successfully.


In [23]:
def apply_email_text_validation(sheet_id, start_row, end_row, start_col, end_col):
    """Restricts text inputs to valid email structures using regular expressions."""
    
    request_body = {
        "requests": [
            {
                "setDataValidation": {
                    "range": {
                        "sheetId": sheet_id,
                        "startRowIndex": start_row,
                        "endRowIndex": end_row,
                        "startColumnIndex": start_col,
                        "endColumnIndex": end_col
                    },
                    "rule": {
                        "condition": {
                            "type": "TEXT_IS_VALID_EMAIL"
                        },
                        "inputMessage": "Please enter a correctly structured email address.",
                        "strict": True
                    }
                }
            }
        ]
    }
    
    sheets.batchUpdate(spreadsheetId=SPREADSHEET_ID, body=request_body).execute()
    print("✅ Email structure validation rules active on targeted column.")

# Example Test: Force valid email strings on Column B (Index 1), rows 2 through 20
apply_email_text_validation(TARGET_TAB_ID, start_row=1, end_row=20, start_col=1, end_col=2)

HttpError: <HttpError 400 when requesting https://sheets.googleapis.com/v4/spreadsheets/1btqCNcTbWSyF8ahVr2DTOcLVukne5nWTUWTLN9xePGU:batchUpdate?alt=json returned "Invalid value at 'requests[0].set_data_validation.rule.condition.type' (type.googleapis.com/google.apps.sheets.v4.BooleanCondition.ConditionType), "TEXT_IS_VALID_EMAIL"". Details: "[{'@type': 'type.googleapis.com/google.rpc.BadRequest', 'fieldViolations': [{'field': 'requests[0].set_data_validation.rule.condition.type', 'description': 'Invalid value at \'requests[0].set_data_validation.rule.condition.type\' (type.googleapis.com/google.apps.sheets.v4.BooleanCondition.ConditionType), "TEXT_IS_VALID_EMAIL"'}]}]">

Complete Error handling and validation during insertion

In [22]:
import pandas as pd
from Google import Create_Service
from googleapiclient.errors import HttpError

# -------------------------------------------------------------------------
# 1. INITIALIZATION & CONFIGURATION
# -------------------------------------------------------------------------
CLIENT_SECRET_FILE = 'client_secret.json'
API_NAME = 'sheets'
API_VERSION = 'v4'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

# Global spreadsheet targets
SPREADSHEET_ID = "1btqCNcTbWSyF8ahVr2DTOcLVukne5nWTUWTLN9xePGU"  # <-- Swap with your real working Sheet ID
TARGET_RANGE = "Sheet1!A:F"                       # Tracks columns: Name, Email, Phone, Company, Source, Status

try:
    service = Create_Service(CLIENT_SECRET_FILE, API_NAME, API_VERSION, scopes=SCOPES)
    sheets = service.spreadsheets()
except Exception as auth_error:
    print(f"🛑 CRITICAL AUTHENTICATION FAILURE: Check your client_secret.json. Details: {auth_error}")

# -------------------------------------------------------------------------
# 2. LOCAL DATA VALIDATION ENGINE (PANDAS)
# -------------------------------------------------------------------------
def validate_and_clean_leads(raw_input_dict):
    """
    Validates data frames before they touch the Google API.
    Separates good rows from bad rows, reporting errors line-by-line.
    """
    df = pd.DataFrame(raw_input_dict)
    
    # Standardize string inputs to prevent case or whitespace mismatches
    df['Lead Source'] = df['Lead Source'].astype(str).str.strip().str.lower()
    df['Status'] = df['Status'].astype(str).str.strip().str.lower()
    
    # Define business rules
    ALLOWED_SOURCES = ["website", "linkedin", "cold_outreach", "referral", "google_ads"]
    ALLOWED_STATUSES = ["new", "contacted", "qualified", "nurturing", "lost"]
    
    # Build strict validation rules
    email_rule = df['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)
    source_rule = df['Lead Source'].isin(ALLOWED_SOURCES)
    status_rule = df['Status'].isin(ALLOWED_STATUSES)
    name_rule = df['Lead Name'].astype(str).str.strip().str.len() > 0
    
    # Combine rules into a master mask
    valid_mask = email_rule & source_rule & status_rule & name_rule
    
    clean_df = df[valid_mask].copy()
    quarantine_df = df[~valid_mask].copy()
    
    # Generate a descriptive error report for bad rows
    if not quarantine_df.empty:
        print("\n❌ DATA VALIDATION REJECTIONS DETECTED:")
        for idx, row in quarantine_df.iterrows():
            reasons = []
            if not name_rule[idx]: reasons.append("Lead Name cannot be blank")
            if not email_rule[idx]: reasons.append(f"Invalid email format ('{row['Email']}')")
            if not source_rule[idx]: reasons.append(f"Invalid Source ('{row['Lead Source']}')")
            if not status_rule[idx]: reasons.append(f"Invalid Status ('{row['Status']}')")
            print(f"  • Row Index {idx} [{row['Lead Name']}]: Rejected due to -> {', '.join(reasons)}")
            
    return clean_df

# -------------------------------------------------------------------------
# 3. API STORAGE ENGINE WITH ERROR HANDLING
# -------------------------------------------------------------------------
def safe_append_leads(spreadsheet_id, target_range, dataframe_payload):
    """
    Safely appends a Pandas DataFrame to Google Sheets.
    Uses detailed error handling to catch and report specific API failures.
    """
    if dataframe_payload.empty:
        print("\n⚠️ Insertion Halted locally: No clean rows passed validation filters.")
        return
        
    # Convert clean rows to the 2D array list format Google requires
    matrix_values = dataframe_payload.astype(str).values.tolist()
    body_content = {'values': matrix_values}
    
    try:
        print(f"\n📡 Attempting to transmit {len(matrix_values)} rows to Google Sheets...")
        response = sheets.values().append(
            spreadsheetId=spreadsheet_id,
            range=target_range,
            valueInputOption="USER_ENTERED",
            insertDataOption="INSERT_ROWS",
            body=body_content
        ).execute()
        
        print("\n" + "🎉"*5 + " SUCCESSFUL API TRANSMISSION " + "🎉"*5)
        print(f"Target Range Appended: {response.get('updates', {}).get('updatedRange')}")
        print(f"Confirmed Rows Inserted: {response.get('updates', {}).get('updatedRows')}")
        
    except HttpError as google_error:
        # Extract details from the API error response
        status_code = google_error.resp.status
        error_details = google_error.error_details if hasattr(google_error, 'error_details') else str(google_error)
        
        print("\n" + "🛑"*5 + " GOOGLE SHEETS API ERROR DETECTED " + "🛑"*5)
        if status_code == 403:
            print("Status 403 (Forbidden): Your account lacks write access to this sheet.")
            print("👉 Fix: Make sure your OAuth login user can edit this file.")
        elif status_code == 404:
            print("Status 404 (Not Found): The Spreadsheet ID or Sheet name does not exist.")
            print(f"👉 Fix: Double check SPREADSHEET_ID and your range tab name ('{target_range.split('!')[0]}').")
        elif status_code == 429:
            print("Status 429 (Rate Limit Exceeded): You are hitting Google's API speed limit restrictions.")
            print("👉 Fix: Add a pause (`time.sleep`) between your code runs.")
        else:
            print(f"HTTP Error Status Code: {status_code}")
            print(f"Raw Details: {error_details}")
            
    except ConnectionError:
        print("\n🛑 NETWORK FAILURE: Could not establish a connection to Google's servers.")
        print("👉 Fix: Check your local internet connection and try running the cell again.")
    except Exception as generic_system_error:
        print(f"\n🛑 UNEXPECTED PYTHON RUNTIME EXCEPTION: {generic_system_error}")

# -------------------------------------------------------------------------
# 4. EXECUTE RUNTIME PIPELINE TEST
# -------------------------------------------------------------------------

# Mixed batch: 2 clean entries, 2 broken entries
inbound_lead_batch = {
    'Lead Name': ['Peter Parker', '  ', 'Bruce Wayne', 'Arthur Dent'],
    'Email': ['spidey@dailybugle.com', 'bad_email.com', 'bruce@wayne.corp', 'arthur@galaxy.org'],
    'Phone': ['555-0900', '555-1111', '555-1000', '42'],
    'Company': ['Daily Bugle', 'Oscorp', 'Wayne Enterprises', 'Hitchhiker Inc'],
    'Lead Source': ['website', 'linkedin', 'FACEBOOK_ADS', 'referral'], # FACEBOOK_ADS is an unapproved source
    'Status': ['new', 'new', 'qualified', 'PANICKED']                   # PANICKED is an unapproved status
}

# Run local validation check
validated_dataframe = validate_and_clean_leads(inbound_lead_batch)

print("\n--- Rows Approved for Sync ---")
display(validated_dataframe)

# Safely push approved entries to your live sheet
safe_append_leads(SPREADSHEET_ID, TARGET_RANGE, validated_dataframe)

🎉 sheets vv4 service created successfully.

❌ DATA VALIDATION REJECTIONS DETECTED:
  • Row Index 1 [  ]: Rejected due to -> Lead Name cannot be blank, Invalid email format ('bad_email.com')
  • Row Index 2 [Bruce Wayne]: Rejected due to -> Invalid Source ('facebook_ads')
  • Row Index 3 [Arthur Dent]: Rejected due to -> Invalid Status ('panicked')

--- Rows Approved for Sync ---


,Lead Name,Email,Phone,Company,Lead Source,Status
0,Peter Parker,spidey@dailybugle.com,555-0900,Daily Bugle,website,new



📡 Attempting to transmit 1 rows to Google Sheets...

🎉🎉🎉🎉🎉 SUCCESSFUL API TRANSMISSION 🎉🎉🎉🎉🎉
Target Range Appended: Sheet1!A7:F7
Confirmed Rows Inserted: 1


(with phone number validation)

In [24]:
def validate_and_clean_leads(raw_input_dict):
    df = pd.DataFrame(raw_input_dict)
    
    # 1. Normalize string inputs
    df['Lead Name'] = df['Lead Name'].astype(str).str.strip()
    df['Email'] = df['Email'].astype(str).str.strip()
    df['Phone'] = df['Phone'].astype(str).str.strip()
    df['Lead Source'] = df['Lead Source'].astype(str).str.strip().str.lower()
    df['Status'] = df['Status'].astype(str).str.strip().str.lower()
    
    # 2. Define Allowed Category Lists
    ALLOWED_SOURCES = ["website", "linkedin", "cold_outreach", "referral", "google_ads"]
    ALLOWED_STATUSES = ["new", "contacted", "qualified", "nurturing", "lost"]
    
    # 3. Apply Strict Compliance Rules
    name_rule = df['Lead Name'].str.len() > 0
    email_rule = df['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)
    source_rule = df['Lead Source'].isin(ALLOWED_SOURCES)
    status_rule = df['Status'].isin(ALLOWED_STATUSES)
    
    # 4. PHONE NUMBER RULE: 3 digits, a dash, then 4 digits (7 digits total, one dash)
    phone_rule = df['Phone'].str.contains(r'^\d{3}-\d{4}$', regex=True, na=False)
    
    # Combine all rules into the master mask
    valid_mask = name_rule & email_rule & source_rule & status_rule & phone_rule
    
    clean_df = df[valid_mask].copy()
    quarantine_df = df[~valid_mask].copy()
    
    # Detailed Console Reporting for Rejections
    if not quarantine_df.empty:
        print("\n❌ LOCAL VALIDATION ALERT: Rejections Detected:")
        for idx, row in quarantine_df.iterrows():
            errors = []
            if not name_rule[idx]: errors.append("Blank Name")
            if not email_rule[idx]: errors.append("Malformed Email")
            if not source_rule[idx]: errors.append(f"Unapproved Source ('{row['Lead Source']}')")
            if not status_rule[idx]: errors.append(f"Unapproved Status ('{row['Status']}')")
            if not phone_rule[idx]: errors.append(f"Invalid Phone Format ('{row['Phone']}'). Must be XXX-XXXX")
            
            print(f"  • Row {idx} [{row['Lead Name']}] rejected due to: {', '.join(errors)}")
            
    return clean_df

In [32]:
test_batch = {
    'Lead Name': ['Tony Stark', 'Bruce Banner', 'Steve Rogers', "@#$"],
    'Email': ['tony@stark.com', 'bruce@gamma.com', 'cap@avengers.org', "898989"], 
    'Phone': ['555-0100', '5550199', '123-456-7890', "xyzabc"], # Banner has no dash, Rogers has too many digits
    'Company': ['Stark Industries', 'Gamma Labs', 'US Army', "valornat"],
    'Lead Source': ['linkedin', 'website', 'referral', "@123"], 
    'Status': ['new', 'new', 'new', "lost"]                 
}

validated_df = validate_and_clean_leads(test_batch)


❌ LOCAL VALIDATION ALERT: Rejections Detected:
  • Row 1 [Bruce Banner] rejected due to: Invalid Phone Format ('5550199'). Must be XXX-XXXX
  • Row 2 [Steve Rogers] rejected due to: Invalid Phone Format ('123-456-7890'). Must be XXX-XXXX
  • Row 3 [@#$] rejected due to: Malformed Email, Unapproved Source ('@123'), Invalid Phone Format ('xyzabc'). Must be XXX-XXXX


Testing and pushing only the clean reocrds

In [26]:
# 1. This is your raw, dirty data source batch
raw_batch = {
    'Lead Name':   ['Natasha Romanoff', 'Clint Barton', 'Wanda Maximoff'],
    'Email':       ['natasha@shield.gov', 'clint@avengers.org', 'wanda_at_westview.io'], # Wanda has a bad email format
    'Phone':       ['555-0300', '5550400', '555-0500'],                                # Clint has a bad phone format
    'Company':     ['S.H.I.E.L.D.', 'Avengers Academy', 'Independent'],
    'Lead Source': ['cold_outreach', 'referral', 'website'],
    'Status':      ['new', 'contacted', 'qualified']
}

# 2. Step 1: Run local validation filter
print("Step 1: Splitting data batch based on validation rules...")
clean_df = validate_and_clean_leads(raw_batch)

# 3. Step 2: Push ONLY the clean records to Google Sheets
print("\nStep 2: Sending approved data to your 'sheets' service...")
safe_append_leads(SPREADSHEET_ID, TARGET_RANGE, clean_df)

Step 1: Splitting data batch based on validation rules...

❌ LOCAL VALIDATION ALERT: Rejections Detected:
  • Row 1 [Clint Barton] rejected due to: Invalid Phone Format ('5550400'). Must be XXX-XXXX
  • Row 2 [Wanda Maximoff] rejected due to: Malformed Email

Step 2: Sending approved data to your 'sheets' service...

📡 Attempting to transmit 1 rows to Google Sheets...

🎉🎉🎉🎉🎉 SUCCESSFUL API TRANSMISSION 🎉🎉🎉🎉🎉
Target Range Appended: Sheet1!A8:F8
Confirmed Rows Inserted: 1


Fetching Function

In [39]:
import pandas as pd
from Google import Create_Service
from googleapiclient.errors import HttpError

# Initialize your exact service variables
CLIENT_SECRET_FILE = 'client_secret.json'
API_NAME = 'sheets'
API_VERSION = 'v4'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
SPREADSHEET_ID = "1btqCNcTbWSyF8ahVr2DTOcLVukne5nWTUWTLN9xePGU"
TARGET_RANGE = "Sheet1!A2:F2"  # Fetching columns A through F

service = Create_Service(CLIENT_SECRET_FILE, API_NAME, API_VERSION, scopes=SCOPES)
sheets = service.spreadsheets()

def safe_fetch_and_validate_crm(spreadsheet_id, sheet_range):
    """
    Fetches lead rows safely via the Google Sheets API, handles structural 
    errors, and checks existing records against your validation rules.
    """
    try:
        print(f"📡 Requesting data from range [{sheet_range}]...")
        response = sheets.values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_range
        ).execute()
        
        raw_rows = response.get('values', [])
        
        # Scenario Check: What if the sheet is completely blank?
        if not raw_rows:
            print("⚠️ Fetch Notice: The target sheet range is completely empty.")
            return pd.DataFrame()  # Return an empty dataframe safely
            
        # Extract headers from row 1 and data from subsequent rows
        headers = raw_rows[0]
        data_content = raw_rows[1:]
        
        if not data_content:
            print("⚠️ Fetch Notice: Found headers, but there are zero rows of data underneath.")
            return pd.DataFrame(columns=headers)

        # Build Dataframe safely matching column size structures
        df = pd.DataFrame(data_content, columns=headers)
        print(f"✅ Data successfully fetched. Total records found: {len(df)}")
        
        # LOCAL AUDIT PHASE: Check if the data in the sheet contains errors
        email_check = df['Email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', regex=True, na=False)
        phone_check = df['Phone'].str.contains(r'^\d{3}-\d{4}$', regex=True, na=False)
        
        corrupted_rows = df[~email_check | ~phone_check]
        if not corrupted_rows.empty:
            print(f"⚠️ AUDIT WARNING: Found {len(corrupted_rows)} malformed records currently sitting inside your live Google Sheet!")
            display(corrupted_rows)
            
        return df

    except HttpError as google_error:
        status_code = google_error.resp.status
        print(f"🛑 GOOGLE API FETCH FAILURE (Status {status_code})")
        if status_code == 404:
            print("👉 Check if your SPREADSHEET_ID is correct or if the tab name is spelled right.")
        elif status_code == 403:
            print("👉 Access Denied: Your client token does not have view rights to this document.")
        return None

# Test the Fetch Engine
crm_df = safe_fetch_and_validate_crm(SPREADSHEET_ID, TARGET_RANGE)

🎉 sheets vv4 service created successfully.
📡 Requesting data from range [Sheet1!A2:F2]...
⚠️ Fetch Notice: Found headers, but there are zero rows of data underneath.


In [40]:
crm_df

,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted


Row Fetch or Look Up based on query

In [41]:
import pandas as pd
from Google import Create_Service
from googleapiclient.errors import HttpError

# Initialize your standard authentication variables
CLIENT_SECRET_FILE = 'client_secret.json'
API_NAME = 'sheets'
API_VERSION = 'v4'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
SPREADSHEET_ID = "1btqCNcTbWSyF8ahVr2DTOcLVukne5nWTUWTLN9xePGU"  # <-- Swap with your real working Sheet ID
TARGET_RANGE = "Sheet1!A:F"                       

service = Create_Service(CLIENT_SECRET_FILE, API_NAME, API_VERSION, scopes=SCOPES)
sheets = service.spreadsheets()

def find_leads_by_criteria(search_column=None, search_value=None, row_number=None):
    """
    Downloads data from the sheet and runs a flexible, localized search engine.
    
    Parameters:
        search_column (str): The column header name to filter by (e.g., 'Lead Name', 'Lead Source', 'Email')
        search_value (str): The specific text value you are looking for (e.g., 'Tony Stark', 'linkedin')
        row_number (int): The literal spreadsheet row number to fetch directly (e.g., 2, 5)
    """
    try:
        # 1. Fetch the raw matrix from Google Sheets
        response = sheets.values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=TARGET_RANGE
        ).execute()
        
        raw_rows = response.get('values', [])
        if not raw_rows:
            print("⚠️ The target sheet is completely empty.")
            return None
            
        headers = raw_rows[0]
        data_rows = raw_rows[1:]
        
        # 2. Convert to Pandas DataFrame for advanced lookup features
        df = pd.DataFrame(data_rows, columns=headers)
        
        # Map out actual spreadsheet row numbers (Row 1 was headers, so Index 0 = Row 2)
        df.index = range(2, len(df) + 2)
        
        # ---------------------------------------------------------
        # LOOKUP SCENARIO A: Fetch by Exact Spreadsheet Row Number
        # ---------------------------------------------------------
        if row_number is not None:
            if row_number in df.index:
                print(f"🎯 Displaying data found strictly on Sheet Row {row_number}:")
                return df.loc[[row_number]]
            else:
                print(f"❌ Error: Row number {row_number} is out of bounds for current sheet data.")
                return None
                
        # ---------------------------------------------------------
        # LOOKUP SCENARIO B: Fetch by Column Name & Target Value
        # ---------------------------------------------------------
        if search_column and search_value:
            # Clean up the column name input to prevent case mismatches
            matched_headers = [h for h in headers if h.strip().lower() == search_column.strip().lower()]
            
            if not matched_headers:
                print(f"❌ Error: Column '{search_column}' not found. Available headers: {headers}")
                return None
                
            actual_header = matched_headers[0]
            
            # Case-insensitive data search evaluation
            matching_mask = df[actual_header].astype(str).str.strip().str.lower() == str(search_value).strip().lower()
            results_df = df[matching_mask]
            
            print(f"🔍 Search matches found for [{actual_header} = '{search_value}']: {len(results_df)}")
            return results_df

        # ---------------------------------------------------------
        # LOOKUP SCENARIO C: No specific filter passed -> Return entire database
        # ---------------------------------------------------------
        print("📋 No search criteria specified. Returning complete CRM snapshot:")
        return df

    except HttpError as err:
        print(f"🛑 Google API Fetch Error: {err}")
        return None

🎉 sheets vv4 service created successfully.


In [42]:
# Looks up any row where the 'Lead Name' matches 'Tony Stark'
results = find_leads_by_criteria(search_column="Lead Name", search_value="Tony Stark")
display(results)

🔍 Search matches found for [Lead Name = 'Tony Stark']: 2


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new


In [43]:
# Looks up all records across your spreadsheet that came from a specific channel
linkedin_leads = find_leads_by_criteria(search_column="Lead Source", search_value="linkedin")
display(linkedin_leads)

🔍 Search matches found for [Lead Source = 'linkedin']: 2


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new


In [44]:
# Instantly flags and grabs a customer record based on their unique email key
specific_email = find_leads_by_criteria(search_column="Email", search_value="pepper@stark.com")
display(specific_email)

🔍 Search matches found for [Email = 'pepper@stark.com']: 0


,Lead Name,Email,Phone,Company,Lead Source,Status


In [ ]:
# If you need to inspect whatever data is currently sitting exactly on Row 3
row_three_data = find_leads_by_criteria(search_column="Email")
display(row_three_data)

📋 No search criteria specified. Returning complete CRM snapshot:


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
3,Clint Barton,hawkeye@avengers.org,555-0400,Avengers Academy,referral,nurturing
4,Wanda Maximoff,wanda@westview.io,555-0500,Independent,website,qualified
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new
6,Bruce Banner,hulk@avengers.org,555-0200,Gamma Labs,website,contacted
7,Peter Parker,spidey@dailybugle.com,555-0900,Daily Bugle,website,new
8,Natasha Romanoff,natasha@shield.gov,555-0300,S.H.I.E.L.D.,cold_outreach,new


In [46]:
col_three_data = find_leads_by_criteria(search_column=3)
display(col_three_data)

📋 No search criteria specified. Returning complete CRM snapshot:


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
3,Clint Barton,hawkeye@avengers.org,555-0400,Avengers Academy,referral,nurturing
4,Wanda Maximoff,wanda@westview.io,555-0500,Independent,website,qualified
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new
6,Bruce Banner,hulk@avengers.org,555-0200,Gamma Labs,website,contacted
7,Peter Parker,spidey@dailybugle.com,555-0900,Daily Bugle,website,new
8,Natasha Romanoff,natasha@shield.gov,555-0300,S.H.I.E.L.D.,cold_outreach,new


In [48]:
import pandas as pd
from Google import Create_Service
from googleapiclient.errors import HttpError

# Global variables linked to your established setup
CLIENT_SECRET_FILE = 'client_secret.json'
API_NAME = 'sheets'
API_VERSION = 'v4'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']  # <-- Swap with your real working Sheet ID
TARGET_RANGE = "Sheet1!A:F"                       

service = Create_Service(CLIENT_SECRET_FILE, API_NAME, API_VERSION, scopes=SCOPES)
sheets = service.spreadsheets()

def advanced_crm_fetch(search_target=None, search_value=None, row_number=None):
    """
    An flexible lookup engine that lets you query data by row numbers, 
    column letters, column numbers, column names, phone numbers, or statuses.
    
    Parameters:
        search_target (str/int): Can be a Column Name ('Phone', 'Status'), 
                                 a Column Letter ('C', 'F'), or a 1-Based Column Number (3, 6).
        search_value (str): The specific value to look for (e.g., '555-0100', 'qualified').
        row_number (int): A specific spreadsheet row number to fetch directly (e.g., 4).
    """
    try:
        # 1. Fetch raw data matrix from Google Sheets
        response = sheets.values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=TARGET_RANGE
        ).execute()
        
        raw_rows = response.get('values', [])
        if not raw_rows:
            print("⚠️ The target sheet is completely empty.")
            return None
            
        headers = raw_rows[0]
        data_rows = raw_rows[1:]
        
        # 2. Build local Pandas Dataframe
        df = pd.DataFrame(data_rows, columns=headers)
        
        # Adjust index to match literal physical sheet row numbers (Headers on Row 1)
        df.index = range(2, len(df) + 2)
        
        # -----------------------------------------------------------------
        # MODE 1: Fetch by Direct Sheet Row Number
        # -----------------------------------------------------------------
        if row_number is not None:
            if row_number in df.index:
                print(f"🎯 Displaying data found on Sheet Row {row_number}:")
                return df.loc[[row_number]]
            else:
                print(f"❌ Error: Row {row_number} is out of bounds.")
                return None

        # -----------------------------------------------------------------
        # MODE 2: Fetch by Conditional Filters (Column Name, Letter, or Number)
        # -----------------------------------------------------------------
        if search_target is not None and search_value is not None:
            target_header = None
            
            # Scenario A: User passed a 1-based column integer number (e.g., 3 for Phone)
            if isinstance(search_target, int) or str(search_target).isdigit():
                col_idx = int(search_target) - 1
                if 0 <= col_idx < len(headers):
                    target_header = headers[col_idx]
                else:
                    print(f"❌ Error: Column index {search_target} is out of range.")
                    return None
            
            # Scenario B: User passed a Column Letter string (e.g., 'C' or 'f')
            elif len(str(search_target).strip()) == 1 and str(search_target).isalpha():
                letter = str(search_target).strip().upper()
                # Translate 'A'->0, 'B'->1, 'C'->2, etc.
                col_idx = ord(letter) - ord('A')
                if 0 <= col_idx < len(headers):
                    target_header = headers[col_idx]
                else:
                    print(f"❌ Error: Column letter '{letter}' is out of mapped data range.")
                    return None
            
            # Scenario C: User passed a textual Header Name (e.g., 'Phone', 'Status')
            else:
                matched_headers = [h for h in headers if h.strip().lower() == str(search_target).strip().lower()]
                if matched_headers:
                    target_header = matched_headers[0]
                else:
                    print(f"❌ Error: Column key '{search_target}' not found. Available: {headers}")
                    return None
            
            # 3. Apply the filter dynamically
            matching_mask = df[target_header].astype(str).str.strip().str.lower() == str(search_value).strip().lower()
            results_df = df[matching_mask]
            
            print(f"🔍 Search matches found for Column [{target_header}] = '{search_value}': {len(results_df)}")
            return results_df

        # Default fallback: Return everything if no parameters passed
        return df

    except HttpError as err:
        print(f"🛑 Google API Error: {err}")
        return None

🎉 sheets vv4 service created successfully.


In [49]:
# Looks up a specific lead based on their phone number
phone_search = advanced_crm_fetch(search_target="Phone", search_value="555-0100")
display(phone_search)

🔍 Search matches found for Column [Phone] = '555-0100': 2


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new


In [50]:
# Looks up data using the 1-based column position index number
status_search_by_num = advanced_crm_fetch(search_target=6, search_value="qualified")
display(status_search_by_num)

🔍 Search matches found for Column [Status] = 'qualified': 1


,Lead Name,Email,Phone,Company,Lead Source,Status
4,Wanda Maximoff,wanda@westview.io,555-0500,Independent,website,qualified


In [51]:
# Looks up data using the standard spreadsheet column letter
status_search_by_letter = advanced_crm_fetch(search_target="F", search_value="new")
display(status_search_by_letter)

🔍 Search matches found for Column [Status] = 'new': 3


,Lead Name,Email,Phone,Company,Lead Source,Status
5,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,new
7,Peter Parker,spidey@dailybugle.com,555-0900,Daily Bugle,website,new
8,Natasha Romanoff,natasha@shield.gov,555-0300,S.H.I.E.L.D.,cold_outreach,new


In [52]:
# Looks up data by passing the column text directly
status_search_by_name = advanced_crm_fetch(search_target="Status", search_value="contacted")
display(status_search_by_name)

🔍 Search matches found for Column [Status] = 'contacted': 2


,Lead Name,Email,Phone,Company,Lead Source,Status
2,Tony Stark,tony@stark.com,555-0100,Stark Industries,linkedin,contacted
6,Bruce Banner,hulk@avengers.org,555-0200,Gamma Labs,website,contacted


Updating

In [30]:
def safe_update_lead_row(spreadsheet_id, sheet_name, row_number, updated_lead_dict):
    """
    Validates updated lead details locally, and if clean, updates 
    the specific row coordinate inside your sheet.
    
    Parameters:
        row_number (int): The actual spreadsheet row number to overwrite (e.g., 2, 3, 4)
    """
    # 1. Local Pre-validation Check
    phone_input = updated_lead_dict.get('Phone', '')
    email_input = updated_lead_dict.get('Email', '')
    
    import re
    if not re.match(r'^\d{3}-\d{4}$', phone_input):
        print(f"❌ UPDATE CANCELLED: Phone '{phone_input}' fails formatting verification (must be XXX-XXXX).")
        return
    if not re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', email_input):
        print(f"❌ UPDATE CANCELLED: Email '{email_input}' fails layout checks.")
        return

    # 2. Map payload into row vector format
    row_vector = [[
        updated_lead_dict.get('Lead Name'),
        email_input,
        phone_input,
        updated_lead_dict.get('Company'),
        updated_lead_dict.get('Lead Source').lower(),
        updated_lead_dict.get('Status').lower()
    ]]
    
    # Target concrete coordinates (e.g., "Sheet1!A3:F3")
    specific_range = f"{sheet_name}!A{row_number}:F{row_number}"
    
    payload = {"values": row_vector}
    
    try:
        print(f"📡 Sending safe update request to row coordinate A{row_number}:F{row_number}...")
        response = sheets.values().update(
            spreadsheetId=spreadsheet_id,
            range=specific_range,
            valueInputOption="USER_ENTERED",
            body=payload
        ).execute()
        
        print(f"🎉 Row {row_number} updated successfully! Cells modified: {response.get('updatedCells')}")
        
    except HttpError as google_error:
        print(f"🛑 GOOGLE API UPDATE FAILURE: Row could not be written. Details: {google_error}")

# --- RUNNING AN UPDATE TEST SCENARIO ---

# Test Case A: Trying to execute an update containing a broken phone format number
broken_update_data = {
    'Lead Name': 'Tony Stark', 'Email': 'tony@stark.com', 'Phone': '12345678', # No dash
    'Company': 'Stark Industries', 'Lead Source': 'linkedin', 'Status': 'qualified'
}
safe_update_lead_row(SPREADSHEET_ID, "Sheet1", row_number=2, updated_lead_dict=broken_update_data)



❌ UPDATE CANCELLED: Phone '12345678' fails formatting verification (must be XXX-XXXX).


In [31]:

# Test Case B: Executing a perfectly valid update to clean up Row 2
clean_update_data = {
    'Lead Name': 'Tony Stark', 'Email': 'tony@stark.com', 'Phone': '555-0100', # Correct format
    'Company': 'Stark Industries', 'Lead Source': 'linkedin', 'Status': 'contacted'
}
safe_update_lead_row(SPREADSHEET_ID, "Sheet1", row_number=2, updated_lead_dict=clean_update_data)

📡 Sending safe update request to row coordinate A2:F2...
🎉 Row 2 updated successfully! Cells modified: 6
